# Experiment 1: PBL and Emission Height

### Objective:
- Show how planetary boundary layer (PBL) dynamics influence pollutant dispersion.
- Demonstrate the role of emission height in determining surface vs. upper-level pollution.
### Simulation Design:
- Select a point source (e.g., power plant or industrial facility).
- Run one simulation with two tracers at different emission heights:
  1. Low-level (near surface, ~50 m)
  2. Elevated (stack emission, ~500 m)
- Compare the dispersion patterns and near-surface concentrations.
### Analysis:
- Plot vertical and horizontal pollutant distributions.
- Analyze how nighttime stable boundary layers trap pollutants near the surface.
- Show how daytime convection enhances vertical mixing.

## Run the experiment
### 1. Link input data and model executable
Insert the directory to data and model.

In [ ]:
# base directory for ICON sources and binary:
export ICONDIR=/Path/to/ICON-model

# directory with initial conditions, external parameters and grid file
export INDIR=/Path/to/Input-data

# directory with xml files
export XMLDIR=/Path/to/xmlfiles

# absolute path to directory with plenty of space:
export OUTDIR=/Path/to/output-directoy

In [ ]:
# create output directory if not existing yet:
if [ ! -d $OUTDIR ]; then
    mkdir -p $OUTDIR
fi
cd $OUTDIR

# copy icon binary to OUTDIR
cp ${ICONDIR}/bin/icon icon.exe

In [ ]:
# reduced radiation grid file
ln -sf ${INDIR}/lam_test_DOM01.parent.nc lam_test_DOM01.parent.nc
# limited area grid
ln -sf ${INDIR}/lam_test_DOM01.nc lam_test_DOM01.nc
# lateral boundary grid
ln -sf ${INDIR}/lateral_boundary.grid.nc lateral_boundary.grid.nc
# external parameter
ln -sf ${INDIR}/lam_test_DOM01.extpar.nc lam_test_DOM01.extpar.nc
# initial  data
ln -sf $INDIR/igfff00000000.nc igfff00000000.nc

# Link files needed for physics
ln -sf ${ICONDIR}/data/ECHAM6_CldOptProps.nc ECHAM6_CldOptProps.nc
ln -sf ${ICONDIR}/data/rrtmg_lw.nc rrtmg_lw.nc

# Dictionary for the mapping: DWD GRIB2 names <-> ICON internal names
ln -sf ${INDIR}/ana_varnames_map_file.txt map_file.ana
# Dictionary for the mapping: GRIB2/Netcdf input names <-> ICON internal names
ln -sf ${INDIR}/map_file.latbc .

### 2. Runscript
No changes necesaary here.

In [ ]:
# ----------------------------------------------------------------------------
# create ICON master namelist
# ----------------------------------------------------------------------------

cat > icon_master.namelist << EOF

! master_nml: ----------------------------------------------------------------
&master_nml
 lrestart                   =                      .FALSE.        ! .TRUE.=current experiment is resumed
/
! master_model_nml: repeated for each model ----------------------------------
&master_model_nml
 model_type                  =                          1         ! identifies which component to run (atmosphere,ocean,...)
 model_name                  =                      "ATMO"        ! character string for naming this component.
 model_namelist_filename     =              "NAMELIST_NWP"        ! file name containing the model namelists
 model_min_rank              =                          1         ! start MPI rank for this model
 model_max_rank              =                      65536         ! end MPI rank for this model
 model_inc_rank              =                          1         ! stride of MPI ranks
/
! time_nml: specification of date and time------------------------------------
&time_nml
 ini_datetime_string         =      "2018-05-07T00:00:00Z"        ! initial date and time of the simulation
 end_datetime_string         =      "2018-05-08T00:00:00Z"        ! end date and time of the simulation
/
EOF

In [ ]:
# ----------------------------------------------------------------------
# model namelists
# ----------------------------------------------------------------------


cat > NAMELIST_NWP << EOF
! parallel_nml: MPI parallelization -------------------------------------------
&parallel_nml
 nproma                      =                          8         ! loop chunk length
 p_test_run                  =                     .FALSE.        ! .TRUE. means verification run for MPI parallelization
 num_io_procs                =                          2         ! number of I/O processors
 num_restart_procs           =                          0         ! number of restart processors
 num_prefetch_proc           =                          1         ! number of processors for LBC prefetching
 iorder_sendrecv             =                          3         ! sequence of MPI send/receive calls
/
! run_nml: general switches ---------------------------------------------------
&run_nml
 ltestcase                   =                     .FALSE.        ! real case run
 num_lev                     =                         50         ! number of full levels (atm.) for each domain
 lvert_nest                  =                     .FALSE.        ! no vertical nesting
 dtime                       =                         60.        ! timestep in seconds
 ldynamics                   =                      .TRUE.        ! compute adiabatic dynamic tendencies
 ltransport                  =                      .TRUE.        ! compute large-scale tracer transport
 ntracer                     =                          5         ! number of advected tracers
 iforcing                    =                          3         ! forcing of dynamics and transport by parameterized processes
 msg_level                   =                          7         ! detailed report during integration
 ltimer                      =                      .TRUE.        ! timer for monitoring the runtime of specific routines
 timers_level                =                         10         ! performance timer granularity
 check_uuid_gracefully       =                      .TRUE.        ! give only warnings for non-matching uuids
 output                      =                        "nml"       ! main switch for enabling/disabling components of the model output
 lart                        =                      .TRUE.
/
! diffusion_nml: horizontal (numerical) diffusion ----------------------------
&diffusion_nml
 lhdiff_vn                   =                      .TRUE.        ! diffusion on the horizontal wind field
 lhdiff_temp                 =                      .TRUE.        ! diffusion on the temperature field
 lhdiff_w                    =                      .TRUE.        ! diffusion on the vertical wind field
 hdiff_order                 =                          5         ! order of nabla operator for diffusion
 itype_vn_diffu              =                          1         ! reconstruction method used for Smagorinsky diffusion
 itype_t_diffu               =                          2         ! discretization of temperature diffusion
 hdiff_efdt_ratio            =                         24.0       ! ratio of e-folding time to time step 
 hdiff_smag_fac              =                          0.025     ! scaling factor for Smagorinsky diffusion
/
! dynamics_nml: dynamical core -----------------------------------------------
&dynamics_nml
 iequations                  =                          3         ! type of equations and prognostic variables
 divavg_cntrwgt              =                          0.50      ! weight of central cell for divergence averaging
 lcoriolis                   =                      .TRUE.        ! Coriolis force
/
! extpar_nml: external data --------------------------------------------------
&extpar_nml
 itopo                       =                          1         ! topography (0:analytical)
 extpar_filename             =  'lam_test_DOM<idom>.extpar.nc'    ! filename of external parameter input file
 n_iter_smooth_topo          =                        1,1         ! iterations of topography smoother
 heightdiff_threshold        =                       3000.        ! height difference between neighb. grid points
 hgtdiff_max_smooth_topo     =                   750.,750.        ! see Namelist doc
 heightdiff_threshold        =                 2250.,1500.
/
! initicon_nml: specify read-in of initial state ------------------------------
&initicon_nml
 init_mode                   =                          7         ! start from DWD fg with subsequent vertical remapping 
 lread_ana                   =                     .FALSE.        ! no analysis data will be read
 dwdfg_filename              = "igfff00000000.nc"                 ! initial data filename
 ana_varnames_map_file       =              "map_file.ana"        ! dictionary mapping internal names onto GRIB2 shortNames
 ltile_coldstart             =                      .TRUE.        ! coldstart for surface tiles
 ltile_init                  =                     .FALSE.        ! set it to .TRUE. if FG data originate from run without tiles
/
! grid_nml: horizontal grid --------------------------------------------------
&grid_nml
 dynamics_grid_filename      =  "lam_test_DOM01.nc"               ! array of the grid filenames for the dycore
 radiation_grid_filename     =  "lam_test_DOM01.parent.nc"        ! array of the grid filenames for the radiation model
 dynamics_parent_grid_id     =                          0         ! array of the indexes of the parent grid filenames
 lredgrid_phys               =                      .TRUE.        ! .true.=radiation is calculated on a reduced grid
 lfeedback                   =                      .TRUE.        ! specifies if feedback to parent grid is performed
 l_limited_area              =                      .TRUE.        ! .TRUE. performs limited area run
 ifeedback_type              =                          2         ! feedback type (incremental/relaxation-based)
 start_time                  =                          0.        ! Time when a nested domain starts to be active [s]
/
! gridref_nml: grid refinement settings --------------------------------------
&gridref_nml
 denom_diffu_v               =                        150.        ! denominator for lateral boundary diffusion of velocity
/
! interpol_nml: settings for internal interpolation methods ------------------
&interpol_nml
 nudge_zone_width            =                          8         ! width of lateral boundary nudging zone
 support_baryctr_intp        =                     .FALSE.        ! barycentric interpolation support for output
 nudge_max_coeff             =                       0.07
 nudge_efold_width           =                        2.0
/
! io_nml: general switches for model I/O -------------------------------------
&io_nml
 itype_pres_msl              =                          5         ! method for computation of mean sea level pressure
 itype_rh                    =                          1         ! method for computation of relative humidity
 lmask_boundary              =                      .TRUE.        ! mask out interpolation zone in output
/
! limarea_nml: settings for limited area mode ---------------------------------
&limarea_nml
 itype_latbc                 =                          1         ! 1: time-dependent lateral boundary conditions
 dtime_latbc                 =                       7200.        ! time difference between 2 consecutive boundary data
 nlev_latbc                  =                         60         ! Number of vertical levels in boundary data
 latbc_boundary_grid         =  'lateral_boundary.grid.nc'        ! Grid file defining the lateral boundary
 latbc_path                  =                 '${INDIR}'        ! Absolute path to boundary data
 latbc_varnames_map_file     =            'map_file.latbc'
 latbc_filename              = 'igfff<ddhhmmss>_lbc.nc'           ! boundary data inputfilename
 init_latbc_from_fg          =                     .FALSE.        ! .TRUE.: take lbc for initial time from first guess
/
! lnd_nml: land scheme switches -----------------------------------------------
&lnd_nml
 ntiles                      =                          1         ! number of tiles
 nlev_snow                   =                          3         ! number of snow layers
 lmulti_snow                 =                      .FALSE.       ! .TRUE. for use of multi-layer snow model
 idiag_snowfrac              =                         20         ! type of snow-fraction diagnosis
 lsnowtile                   =                       .TRUE.       ! .TRUE.=consider snow-covered and snow-free separately
 itype_root                  =                          2         ! root density distribution
 itype_heatcond              =                          3         ! type of soil heat conductivity
 itype_lndtbl                =                          4         ! table for associating surface parameters
 itype_evsl                  =                          4         ! type of bare soil evaporation
 itype_root                  =                          2         ! root density distribution
 cwimax_ml                   =                      5.e-4         ! scaling parameter for max. interception storage
 c_soil                      =                       1.75         ! surface area density of the evaporative soil surface
 c_soil_urb                  =                        0.5         ! same for urban areas
 lseaice                     =                      .TRUE.        ! .TRUE. for use of sea-ice model
 llake                       =                      .TRUE.        ! .TRUE. for use of lake model
/
! nonhydrostatic_nml: nonhydrostatic model -----------------------------------
&nonhydrostatic_nml
 iadv_rhotheta               =                          2         ! advection method for rho and rhotheta
 ivctype                     =                          2         ! type of vertical coordinate
 itime_scheme                =                          4         ! time integration scheme
 ndyn_substeps               =                          5         ! number of dynamics steps per fast-physics step
 exner_expol                 =                          0.333     ! temporal extrapolation of Exner function
 vwind_offctr                =                          0.2       ! off-centering in vertical wind solver
 damp_height                 =                      12500.0       ! height at which Rayleigh damping of vertical wind starts
 rayleigh_coeff              =                          1.5       ! Rayleigh damping coefficient
 divdamp_order               =                         24         ! order of divergence damping 
 divdamp_type                =                         3          ! type of divergence damping
 divdamp_fac                 =                          0.004     ! scaling factor for divergence damping
 igradp_method               =                          3         ! discretization of horizontal pressure gradient
 l_zdiffu_t                  =                      .TRUE.        ! specifies computation of Smagorinsky temperature diffusion
 thslp_zdiffu                =                          0.02      ! slope threshold (temperature diffusion)
 thhgtd_zdiffu               =                        125.0       ! threshold of height difference (temperature diffusion)
 htop_moist_proc             =                      22500.0       ! max. height for moist physics
 hbot_qvsubstep              =                      22500.0       ! height above which QV is advected with substepping scheme
/
! nwp_phy_nml: switches for the physics schemes ------------------------------
&nwp_phy_nml
 inwp_gscp                   =                          2         ! cloud microphysics and precipitation
 inwp_convection             =                          1         ! convection
 lshallowconv_only           =                      .TRUE.        ! only shallow convection
 inwp_radiation              =                          4         ! radiation
 inwp_cldcover               =                          1         ! cloud cover scheme for radiation
 inwp_turb                   =                          1         ! vertical diffusion and transfer
 inwp_satad                  =                          1         ! saturation adjustment
 inwp_sso                    =                          1         ! subgrid scale orographic drag
 inwp_gwd                    =                          0         ! non-orographic gravity wave drag
 inwp_surface                =                          1         ! surface scheme
 latm_above_top              =                      .TRUE.        ! take into account atmosphere above model top for radiation computation
 ldetrain_conv_prec          =                      .TRUE.
 efdt_min_raylfric           =                       7200.        ! minimum e-folding time of Rayleigh friction
 itype_z0                    =                          2         ! type of roughness length data
 icapdcycl                   =                          3         ! apply CAPE modification to improve diurnalcycle over tropical land
 icpl_aero_conv              =                          1         ! coupling between autoconversion and Tegen aerosol climatology
 icpl_aero_gscp              =                          1         ! coupling between autoconversion and Tegen aerosol climatology
 lrtm_filename               =                'rrtmg_lw.nc'       ! longwave absorption coefficients for RRTM_LW
 cldopt_filename             =             'rrtm_cldopt.nc'       ! RRTM cloud optical properties
 dt_rad                      =                         720.       ! time step for radiation in s
 dt_conv                     =                 120.,90.,90.       ! time step for convection in s (domain specific)
 dt_sso                      =               120.,360.,360.       ! time step for SSO parameterization
 dt_gwd                      =               360.,360.,360.       ! time step for gravity wave drag parameterization
/
! nwp_tuning_nml: additional tuning parameters ----------------------------------
&nwp_tuning_nml
 itune_albedo                =                          1         ! reduced albedo (w.r.t. MODIS data) over Sahara
 tune_gkwake                 =                        1.8
 tune_gkdrag                 =                        0.01
 tune_minsnowfrac            =                        0.3
/
! output_nml: specifies an output stream --------------------------------------
&output_nml
 filetype                    =                          4 ! output format: 2=GRIB2, 4=NETCDFv2
 dom                         =                          1 ! write domain 1 only
 output_bounds               =        0., 10000000., 3600. ! start, end, increment
 steps_per_file              =                          1 ! number of steps per file
 mode                        =                          1 ! 1: forecast mode (relative t-axis), 2: climate mode (absolute t-axis)
 include_last                =                     .FALSE.
 output_filename             =                    'NWP_LAM'
 filename_format             = '<output_filename>_DOM<physdom>_<jfile>' ! file name base
 output_grid                 =                      .TRUE.
 remap                       =                          1 ! 1: remap to lat-lon grid
 reg_lon_def                 =              2.0,0.1,18.0
 reg_lat_def                 =               45.,0.1,56.0
 ml_varlist                  = 'u', 'v', 'w', 'tke', 'pres', 'z_mc', 'z_ifc', 'temp','theta_v', 'tempv','rho','group:ART_chemistry'
/
! radiation_nml: radiation scheme ---------------------------------------------
&radiation_nml
 irad_o3                     =                          7         ! ozone climatology
 irad_aero                   =                          6         ! aerosols
 ecrad_data_path             =  '${ICONDIR}/externals/ecrad/data'
 albedo_type                 =                          2         ! type of surface albedo
 vmr_co2                     =                    390.e-06
 vmr_ch4                     =                   1800.e-09
 vmr_n2o                     =                   322.0e-09
 vmr_o2                      =                     0.20946
 vmr_cfc11                   =                    240.e-12
 vmr_cfc12                   =                    532.e-12
/
! sleve_nml: vertical level specification -------------------------------------
&sleve_nml
 min_lay_thckn               =                         20.0       ! layer thickness of lowermost layer
 top_height                  =                      23000.0       ! height of model top
 stretch_fac                 =                          0.65      ! stretching factor to vary distribution of model levels
 decay_scale_1               =                       4000.0       ! decay scale of large-scale topography component
 decay_scale_2               =                       2500.0       ! decay scale of small-scale topography component
 decay_exp                   =                          1.2       ! exponent of decay function
 flat_height                 =                      16000.0       ! height above which the coordinate surfaces are flat
/
! transport_nml: tracer transport ---------------------------------------------
&transport_nml
 ivadv_tracer                =              3, 3, 3, 3, 3         ! tracer specific method to compute vertical advection
 itype_hlimit                =              3, 4, 4, 4, 4         ! type of limiter for horizontal transport
 ihadv_tracer                =             52, 2, 2, 2, 2         ! tracer specific method to compute horizontal advection
 llsq_svd                    =                      .TRUE.        ! use SV decomposition for least squares design matrix
/
! turbdiff_nml: turbulent diffusion -------------------------------------------
&turbdiff_nml
 tkhmin                      =                          0.75      ! scaling factor for minimum vertical diffusion coefficient
 tkmmin                      =                          0.75      ! scaling factor for minimum vertical diffusion coefficient
 pat_len                     =                        750.0       ! effective length scale of thermal surface patterns
 c_diff                      =                          0.2       ! length scale factor for vertical diffusion of TKE
 rat_sea                     = 8.5                                ! ** new since r20191: 8.5 for R3B7, 8.0 for R2B6 in order to get similar temperature biases in the tropics **
 rlam_heat                   = 1.0                                ! check before production runs
 ltkesso                     =                        .TRUE.      ! consider TKE-production by sub-grid SSO wakes
 frcsmot                     =                          0.2       ! these 2 switches together apply vertical smoothing of the TKE source terms
 imode_frcsmot               =                            2       ! in the tropics (only), which reduces the moist bias in the tropical lower troposphere
 itype_sher                  =                            3       ! type of shear forcing used in turbulence
 ltkeshs                     =                        .TRUE.      ! include correction term for coarse grids in hor. shear production term
 a_hshr                      =                          2.0       ! length scale factor for separated horizontal shear mode
 icldm_turb                  =                            1       ! mode of cloud water representation in turbulence
 ldiff_qi                    =                        .TRUE.
/
! art_nml: Aerosols and Reactive Trace gases extension
&art_nml
 lart_diag_out   = .TRUE.
 lart_chem       = .TRUE.
 lart_chemtracer = .TRUE.
 lart_pntSrc     = .TRUE.
 cart_pntSrc_xml      = '${XMLDIR}/pntSrc.xml'
 cart_chemtracer_xml  = '${XMLDIR}/tracers_chemtr.xml'
/
EOF

### 3. Job Settings
Insert the correct project account.

In [ ]:
# ----------------------------------------------------------------------
# run the model!
# ----------------------------------------------------------------------

cat > ${OUTDIR}/job_ICON << ENDFILE
#!/bin/bash
#SBATCH --job-name=EXP01_ART          # Specify job name
#SBATCH --partition=compute            # Specify partition name
#SBATCH --nodes=4                    # Specify number of nodes
#SBATCH --ntasks-per-node=128          # Specify number of (MPI) tasks on each node
#SBATCH --time=01:00:00                 # Set a limit on the total run time
#SBATCH --account=????                # Charge resources on this project account

unset SLURM_EXPORT_ENV 
unset SLURM_MEM_PER_NODE
unset SBATCH_EXPORT

export LD_LIBRARY_PATH="/usr/lib:/usr/lib64:/sw/spack-levante/netcdf-c-4.8.1-2k3cmu/lib:/sw/spack-levante/netcdf-fortran-4.5.3-k6xq5g/lib:/sw/spack-levante/hdf5-1.12.1-tvymb5/lib:/sw/spack-levante/eccodes-2.21.0-3ehkbb/lib64:/sw/spack-levante/intel-oneapi-mkl-2022.0.1-ttdktf/mkl/2022.0.1/lib/intel64/"

export OMPI_MCA_pml="ucx"
export OMPI_MCA_btl=self
export OMPI_MCA_osc="pt2pt"
export UCX_IB_ADDR_TYPE=ib_global

export OMPI_MCA_coll="^ml,hcoll"
export OMPI_MCA_coll_hcoll_enable="0"
export HCOLL_ENABLE_MCAST_ALL="0"
export HCOLL_MAIN_IB=mlx5_0:1
export UCX_NET_DEVICES=mlx5_0:1
export UCX_TLS=mm,knem,cma,dc_mlx5,dc_x,self
export UCX_UNIFIED_MODE=y
export HDF5_USE_FILE_LOCKING=FALSE
export OMPI_MCA_io="romio321"
export UCX_HANDLE_ERRORS=bt

ulimit -s 102400
ulimit -c 0

srun -l --cpu_bind=cores --distribution=block:cyclic --propagate=STACK,CORE ./icon.exe

ENDFILE

### 4. Submit the job

In [ ]:
cd $OUTDIR && chmod +x job_ICON && sbatch job_ICON

<div class="alert alert-success">
    <b style="color:#2d4b9b;">Check the slurm-file for errors</b>
    <ul>
      <li>Open an interactive Linux terminal by clicking the following button <button data-commandlinker-command="terminal:create-new">Create Terminal</button></li>
      <li>Go to your output directory using the cd command</li>
      <li>Open the slurm file with e.g., vim</li>
</div>

### 5. Analyze the results
Analize the results with the [Plot_PBL_EmissHeight.ipynb](Plot_PBL_EmissHeight.ipynb) jupyter notebook.